# Initial Route Set Generation

In [1]:
%load_ext autoreload
%reload_ext autoreload

import networkx as nx
import osmnx as ox
import geopandas as gpd
import utm, copy, os, random
import pickle

from shapely.geometry import LineString, Point
from shapely.ops import split
from geopy.distance import geodesic

from genetic_algorithm.schema import StopCandidate, Route
from genetic_algorithm.reverse_routes import get_reverse_route
from utils.route_utils import degrees_to_meters, meters_to_degrees, disable_surrounding_nodes, enable_nodes, add_points_to_graph, get_enabled_node_with_highest_edge_probability
from utils.checkpoint_utils import import_networks
from config.config import MAX_DISTANCE, WALKING_DISTANCES

## Graph and Features

In [ ]:
# NOTE: SELECT THE CITY HERE, COMMENT OUT THE REMAINING CITIES
select_city = "Makati, Philippines"
city_file = "map/Makati.graphml" # FIXME: THIS IS WRONG PATH

# GENERATION OF MAIN CITY GRAPH
# IF FIRST TIME RUNNING, RUN THIS CODE TO GENERATE THE GRAPH
def generate_graph():
    mode = 'drive'
    graph = ox.graph_from_place(select_city, network_type=mode)
    ox.save_graphml(graph, city_file) # Save it as a file

def load_graph():
    graph = ox.load_graphml(city_file)

    print("Graph loaded successfully")
    print("NUMBER OF EDGES: ", graph.number_of_edges())
    print("NUMBER OF NODES: ", graph.number_of_nodes())
    print('\n')
    return graph

# if not os.path.exists(city_file):
#     generate_graph()

# THIS IS THE MAIN GRAPH FOR THE CITY TO BE USED FOR ALL FUNCTIONS
CITY_GRAPH = load_graph()

# Iterate over the edges and assign a unique edge_id
edge_id = 0
for u, v, k in CITY_GRAPH.edges(keys=True):
    CITY_GRAPH[u][v][k]['edge_id'] = edge_id
    edge_id += 1

Graph loaded successfully
NUMBER OF EDGES:  5544
NUMBER OF NODES:  2284




## Initial Filtering

In [5]:
### For Filtering the roads and other features
# GETTING ROADS AND WATERWAYS

road = ox.graph_to_gdfs(CITY_GRAPH,nodes=False, edges=True)
# Get all the roads that are not junctions (ex. Roundabouts, intersection, etc.)
filtered_roads = road[road['junction'].isna()]

# Separate roads whose highway types are only one value and those that are more than 1 (lists)
rows_with_lists = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, list))]
rows_with_strings = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, str))]

# Allowed Roads to place stops
filter_options = ['primary', 'secondary', 'tertiary', 'trunk', 'unclassified']

# To separate zones
separation_options = ['primary', 'secondary', 'tertiary', 'unclassified']

# Get the roads whose widths are above the threshold
def check_list(lst):
    return any(x in filter_options for x in lst)

# Download OpenStreetMap data for the area of interest
waterways = ox.features_from_place(select_city, tags={'waterway': True})
filtered_rivers = waterways[waterways['waterway'].isin(['river'])]
filtered_streams = waterways[waterways['waterway'].isin(['stream'])]

# Get all the roads with the allowed road types
filtered_roads_strings = rows_with_strings.loc[rows_with_strings['highway'].isin(filter_options)]
filtered_roads_lists = rows_with_lists[rows_with_lists['highway'].apply(check_list)]

# Create spatial index
filtered_roads_strings_sindex = filtered_roads_strings.sindex
filtered_roads_lists_sindex = filtered_roads_lists.sindex
filtered_rivers_sindex = filtered_rivers.sindex
filtered_streams_sindex = filtered_streams.sindex

# Total Distance of the allowed road types
TOTAL_NETWORK_ROAD_DISTANCE = 0
for index, row in filtered_roads_strings.iterrows():
    TOTAL_NETWORK_ROAD_DISTANCE += row['length']

## Stop Placement


In [ ]:
# CREATING STOPS
# It should return a list of coordinates/nodes for stop and a graph of stops
# if residential area, check if the population density

# Global Variables used:
# list_of_stops - List of stops
# graph_of_stops - graph of all stops placed
def place_stops_on_roads(amenity_graph, graph_of_stops, list_of_stops):
    list_relevant_edges_set = []
    string_relevant_edges_set = []

    stop_id = 0
    for node_key, node_data in amenity_graph.nodes(data=True):
        # All tranportation points are automatically stops
        if node_data['geometry'].geom_type in ['Point']:
            if node_data['amenity'] == 'transportation':
                # Find the nearest edge to the location point
                nearest_edge = ox.distance.nearest_edges(graph_of_stops, X=node_data['x'], Y=node_data['y'], return_dist=False)
                node1, node2, key = nearest_edge
                edge = graph_of_stops.get_edge_data(node1, node2)
                data = list(edge.values())[0]


                # get the line
                if 'geometry' not in data:
                    gdf_edges = ox.graph_to_gdfs(graph_of_stops, nodes=False, edges=True)
                    line = gdf_edges.loc[nearest_edge]['geometry']
                    edge_length = gdf_edges.loc[nearest_edge]['length']
                else:
                    line = data['geometry']
                    edge_length = data['length']


                if edge_length > 20:
                    # x - lon - 120
                    # y - lat - 14
                    # LineString(lon, lat)
                    # geodesic(lat, lon)

                    print(f"STOP ID: {stop_id}")
                    if node1 < 10000:
                        notice = '!'
                        print(f"SEEN ADDED NODE1{notice} {node1} - {graph_of_stops.nodes[node1]['y']}, {graph_of_stops.nodes[node1]['x']}")

                    if node2 < 10000:
                        notice = '!'
                        print(f"SEEN ADDED NODE2{notice} {node2} - {graph_of_stops.nodes[node2]['y']}, {graph_of_stops.nodes[node2]['x']}")

                    print(f"EDGE NODES ({node1}, {node2})")

                    point_a = (graph_of_stops.nodes[node1]['y'], graph_of_stops.nodes[node1]['x'])
                    point_c = (graph_of_stops.nodes[node2]['y'], graph_of_stops.nodes[node2]['x'])

                    if len(line.coords) == 2:
                        print("Transpo Stop WAY 1")
                        # Get a random position between the line
                        random_position = random.uniform(0.3 * edge_length, edge_length * 0.7)

                        # Convert to UTM
                        node1_utm = utm.from_latlon(point_a[0], point_a[1])
                        node2_utm = utm.from_latlon(point_c[0], point_c[1])

                        utm_line = LineString([(node1_utm[0], node1_utm[1]), (node2_utm[0], node2_utm[1])])

                        # Calculate the coordinate along the edge at the random position
                        point_on_road_utm = calculate_coordinate_along_edge(utm_line, random_position)
                        to_convert = (point_on_road_utm[0], point_on_road_utm[1], node1_utm[2], node1_utm[3])
                        point_on_road = utm.to_latlon(*to_convert)
                        lon = point_on_road[1]
                        lat = point_on_road[0]
                    else:
                        print("Transpo Stop WAY 2")
                        # Accessing all coordinates
                        coordinates = list(line.coords)
                        coordinates = coordinates[1:-1]
                        # Randomly pick one coordinate
                        point_on_road = random.choice(coordinates)
                        lon = point_on_road[0]
                        lat = point_on_road[1]

                    isTranspo = True

                    graph_of_stops.add_node(stop_id, x=lon, y=lat, isTranspo=isTranspo)
                    list_of_stops.append(StopCandidate(lat, lon, True, stop_id, 0))

                    # Get the points
                    point_b = (lat, lon) # y,x

                    # Calculate new geometries
                    if len(line.coords) == 2:
                        line_ab = LineString([(point_a[1], point_a[0]), (point_b[1], point_b[0])])
                        line_bc = LineString([(point_b[1], point_b[0]), (point_c[1], point_c[0])])
                    else:
                        result = split(line, Point(point_b[1], point_b[0]))
                        new_lines = [geom for geom in result.geoms]
                        line_ab = new_lines[0]
                        line_bc = new_lines[1]

                    print("POINT B: ", point_b)

                    print("NEW LINES")
                    print(line_ab)
                    print(line_bc)
                    print(f"ORIGINAL LINE: {line}")
                    print("---------")

                    # Calculate the new distances
                    distance_ab = geodesic(point_a, point_b).meters
                    distance_bc = geodesic(point_b, point_c).meters

                    print("NEW DISTANCES")
                    print(distance_ab)
                    print(distance_bc)
                    print(f"ORIGINAL DISTANCE: {data['length']}")
                    print("---------")


                    # Get the edge data and adjust distances
                    edge_data = data.copy()
                    edge_data['length'] = distance_ab
                    edge_data['geometry'] = line_ab
                    graph_of_stops.add_edge(node1, stop_id, **edge_data)

                    edge_data['length'] = distance_bc
                    edge_data['geometry'] = line_bc
                    graph_of_stops.add_edge(stop_id, node2, **edge_data)

                    print(f"HAS EDGE BETWEEN {node1} and {stop_id} : {graph_of_stops.has_edge(node1, stop_id)}")
                    print(f"HAS EDGE BETWEEN {stop_id} and {node2} : {graph_of_stops.has_edge(stop_id, node2)}")

                    # Remove the original edge
                    graph_of_stops.remove_edge(node1, node2)
                    stop_id += 1
                    print()


        elif node_data['geometry'].geom_type in ['MultiPolygon', 'Polygon']:
            # Get the roads surrounding and inside the node polygons
            buffer_poly = node_data['geometry'].buffer(meters_to_degrees(30))
            relevant_edges = get_relevant_edges(buffer_poly, list_relevant_edges_set, string_relevant_edges_set)

            # Calculate the number of stops based on node size and population density
            node_size = degrees_to_meters(node_data['geometry'].area)

            # Place stops randomly on these roads
            stop_id = place_stops_along_edges(relevant_edges, buffer_poly, node_size, graph_of_stops, list_of_stops, stop_id)


def get_relevant_edges(polygon, list_relevant_edges_set, string_relevant_edges_set):
    relevant_edges = []

    # Check intersection with filtered roads
    possible_matches_roads = filtered_roads_strings.iloc[list(filtered_roads_strings_sindex.intersection(polygon.bounds))]
    for index, row in possible_matches_roads.iterrows():
        if polygon.intersects(row['geometry']) and row['highway'] in filter_options:
            row_edge_id = row['edge_id']
            if row_edge_id not in string_relevant_edges_set:
                relevant_edges.append([index, row])
                string_relevant_edges_set.append(row_edge_id)


    possible_matches_lists = filtered_roads_lists.iloc[list(filtered_roads_lists_sindex.intersection(polygon.bounds))]
    for index, row in possible_matches_lists.iterrows():
        if polygon.intersects(row['geometry']):
            list_highway = row['highway']
            if any(x in filter_options for x in list_highway):
                row_edge_id = row['edge_id']
                if row_edge_id not in list_relevant_edges_set:
                    relevant_edges.append([index, row])
                    list_relevant_edges_set.append(row_edge_id)

    return relevant_edges

def place_stops_along_edges(edges, polygon, node_size, graph_of_stops, list_of_stops, stop_id):
    # Place stops randomly along the edges within the polygon

    if len(edges) > 0:
        for edge in edges:
            # Getting the edge data
            index = edge[0]
            data = edge[1]
            node1 = index[0]
            node2 = index[1]


            # get the line
            if 'geometry' not in data:
                gdf_edges = ox.graph_to_gdfs(graph_of_stops, nodes=False, edges=True)
                line = gdf_edges.loc[(node1, node2, 0)]['geometry']
                edge_length = gdf_edges.loc[(node1, node2, 0)]['length']
            else:
                line = data['geometry']
                edge_length = data['length']

            if edge_length > 20:
                # x - lon - 120
                # y - lat - 14
                # LineString(lon, lat)
                # geodesic(lat, lon)
                print(f"Stop ID: {stop_id}")

                if node1 < 10000:
                        notice = '!'
                        print(f"SEEN ADDED NODE1{notice} ({node1}, {node2}) - {graph_of_stops.nodes[node1]['y']}, {graph_of_stops.nodes[node1]['x']}")

                if node2 < 10000:
                    notice = '!'
                    print(f"SEEN ADDED NODE2{notice} ({node1}, {node2}) - {graph_of_stops.nodes[node2]['y']}, {graph_of_stops.nodes[node2]['x']}")


                # Get the coordinates of the two nodes
                point_a = (graph_of_stops.nodes[node1]['y'], graph_of_stops.nodes[node1]['x'])
                point_c = (graph_of_stops.nodes[node2]['y'], graph_of_stops.nodes[node2]['x'])

                if len(line.coords) == 2:
                    print("Along edges WAY 1")
                    # Get a random position between the line
                    random_position = random.uniform(0.3 * edge_length, edge_length * 0.7)

                    # Convert to UTM
                    node1_utm = utm.from_latlon(point_a[0], point_a[1])
                    node2_utm = utm.from_latlon(point_c[0], point_c[1])

                    utm_line = LineString([(node1_utm[0], node1_utm[1]), (node2_utm[0], node2_utm[1])])

                    # Calculate the coordinate along the edge at the random position
                    point_on_road_utm = calculate_coordinate_along_edge(utm_line, random_position)
                    to_convert = (point_on_road_utm[0], point_on_road_utm[1], node1_utm[2], node1_utm[3])
                    point_on_road = utm.to_latlon(*to_convert)
                    lon = point_on_road[1]
                    lat = point_on_road[0]
                else:
                    print("Along edges WAY 2")
                    # Accessing all coordinates
                    coordinates = list(line.coords)
                    coordinates = coordinates[1:-1]
                    # Randomly pick one coordinate
                    point_on_road = random.choice(coordinates)
                    lon = point_on_road[0]
                    lat = point_on_road[1]

                # Add the stop to graph and list
                isTranspo = False

                graph_of_stops.add_node(stop_id, x=lon, y=lat, isTranspo=isTranspo)
                list_of_stops.append(StopCandidate(lat, lon, False, stop_id, node_size))

                # Get the points
                point_b = (lat, lon)

                # Calculate new geometries

                if len(line.coords) == 2:
                    line_ab = LineString([(point_a[1], point_a[0]), (point_b[1], point_b[0])])
                    line_bc = LineString([(point_b[1], point_b[0]), (point_c[1], point_c[0])])
                else:
                    result = split(line, Point(point_b[1], point_b[0]))
                    new_lines = [geom for geom in result.geoms]
                    line_ab = new_lines[0]
                    line_bc = new_lines[1]

                print("POINT B: ", point_b)

                print("NEW LINES")
                print(line_ab)
                print(line_bc)
                print("---------")

                # Calculate the new distances
                distance_ab = geodesic(point_a, point_b).meters
                distance_bc = geodesic(point_b, point_c).meters

                print("NEW DISTANCES")
                print(distance_ab)
                print(distance_bc)
                print(f"ORIGINAL DISTANCE: {data['length']}")
                print("---------")

                # Add the new distances and new geometries
                edge_data = data.copy()
                edge_data['length'] = distance_ab
                edge_data['geometry'] = line_ab
                graph_of_stops.add_edge(node1, stop_id, **edge_data)

                edge_data['length'] = distance_bc
                edge_data['geometry'] = line_bc
                graph_of_stops.add_edge(stop_id, node2, **edge_data)

                print(f"HAS EDGE BETWEEN {node1} and {stop_id} : {graph_of_stops.has_edge(node1, stop_id)}")
                print(f"HAS EDGE BETWEEN {stop_id} and {node2} : {graph_of_stops.has_edge(stop_id, node2)}")

                # Remove the original edge
                graph_of_stops.remove_edge(node1, node2)
                stop_id += 1
                print()

    return stop_id


def calculate_coordinate_along_edge(edge, position):
    # Calculate the coordinate along the edge at the given position
    point = edge.interpolate(position)
    return point.x, point.y


## Route Connection

In [ ]:
# V2 - Graph with the snapping function
# Generate Route Network from connected routes

# Global Variables used:
# graph_of_stops - Graph of stops that will be used to create routes
def generate_route_network(stop_nodes, max_walking_dist, max_stops, max_routes, graph_of_stops, city_area_sum):
    overall_graph = nx.MultiDiGraph() # The route network graph
    next_nodes = [n for n in stop_nodes]
    enable_nodes(next_nodes)
    route_network = []
    reverse_route_network = []
    num_routes = 0 # Count number of routes

    while num_routes < max_routes:
        route_id = f'{num_routes}-A' # This will be used as a key for the edge
        next_nodes = [n for n in stop_nodes] # Resets the list of nodes so that nodes can be reused in a different
        selected_node = random.choice(next_nodes) # For the first node
        next_nodes.remove(selected_node)
        route_gen, route_length = generate_route(selected_node, next_nodes, max_walking_dist, max_stops, overall_graph, city_area_sum, route_id, graph_of_stops)

        if len(route_gen) > 1:
            new_route = Route(route_gen, route_id, route_length) # Create a new route object to store the Route ID and the route itself

            # Snap it to the graph
            add_to_graph(route_gen, overall_graph, graph_of_stops, route_id)

            # Get the reverse route
            reverse_route = get_reverse_route(new_route, graph_of_stops, overall_graph, max_walking_dist)

            route_network.append(new_route)
            reverse_route_network.append(reverse_route)
            num_routes += 1

    return route_network, overall_graph, reverse_route_network

def add_to_graph(route, overall_graph, graph_of_stops, route_id):

    # Directly add nodes based on node identifiers
    for connection in route:
        overall_graph.add_node(connection[0], **graph_of_stops.nodes[connection[0]]) # The origin
        overall_graph.add_node(connection[-1], **graph_of_stops.nodes[connection[-1]]) # The destination

        distance_travelled = 0
        # Get the total distance from point A to point B
        for i in range(len(connection)-1):
            node_data = graph_of_stops.nodes[connection[i]]
            next_node_data = graph_of_stops.nodes[connection[i+1]]
            distance_travelled += geodesic(node_data['y'], node_data['x'], next_node_data['y'], next_node_data['x']) # TODO: Double check this was not double checked yet since baka mali yung type

        # Finally, Add the edge
        overall_graph.add_edge(connection[0], connection[-1], key=route_id, road_path = connection, distance = distance_travelled) # Add edge


# Generate route from stop nodes
def generate_route(source, next_nodes, max_walking_dist, connection_type, max_stops, network_graph, city_area_sum, route_id, graph_of_stops):
    NODE_YX = {n: (data['y'], data['x']) for n, data in graph_of_stops.nodes(data=True)}

    short_route_list = [] # List of nx.shortest_path results
    totalDistance = 0
    orig_node = source
    num_stops = 0 # Count number of stops

    # CONFIGURATION
    max_tries = 3 # This is the max number of tries before breaking the loop || To avoid longer runtimes
    current_tries = 0

    while totalDistance < MAX_DISTANCE and num_stops < max_stops:

        #print(f"Selected node is {selected_node.getLat()}, {selected_node.getLong()}")
        enable_nodes(next_nodes)
        disable_surrounding_nodes(next_nodes, orig_node, max_walking_dist)
        enabled_nodes = [n for n in next_nodes if n.enabled]
        if len(enabled_nodes) == 0:
            break

        #print(f"{len(enabled_nodes)} nodes out of {len(next_nodes)}")
        dest_node = get_enabled_node_with_highest_edge_probability(orig_node, enabled_nodes, connection_type, city_area_sum, network_graph) # Getting the destination node

        if (dest_node == None or dest_node.id == orig_node.id):
            break

        # Remove it as a candidate
        next_nodes.remove(dest_node)

        # This is to check if there is already an exiting edge in the route. If true, then it should not connect
        connection_edge1 = network_graph.has_edge(orig_node.id, dest_node.id, route_id)
        connection_edge2 = network_graph.has_edge(dest_node.id, orig_node.id, route_id)

        # If there is already an existing edge in either direction for this route_id, do not connect
        if connection_edge1 or connection_edge2:
            current_tries += 1
            if current_tries == max_tries:
                break
            continue  # important: skip trying shortest path

        # Try to get shortest path (cached). If no path exists, treat like nx.has_path == False.
        try:
            shortest_route = nx.shortest_path(graph_of_stops, orig_node.id, dest_node.id, weight='length')
        except nx.NetworkXNoPath:
            current_tries += 1
            if current_tries == max_tries:
                break
            continue

        distance_travelled = 0
        # Get the total distance from point A to point B
        for i in range(len(shortest_route)-1):
           ay, ax = NODE_YX[shortest_route[i]]
           by, bx = NODE_YX[shortest_route[i+1]]
           distance_travelled += geodesic((ay, ax), (by, bx)).kilometers

        # Checks if it does not exceed the max distance
        if totalDistance + distance_travelled <= MAX_DISTANCE:

            # Updating local degree count used for connection probability
            orig_node.degree += 1
            dest_node.degree += 1

            totalDistance += distance_travelled
            short_route_list.append(shortest_route)
            num_stops += 1

            orig_node = dest_node # Now change the origin to the destination
        else:
            break
    if len(short_route_list) > 4 and totalDistance > 7:
        print(f"# OF CONNECTIONS AND TOTAL DISTANCE: {len(short_route_list)} - {totalDistance}")
        return short_route_list, totalDistance

    else:
        return [], []

In [ ]:
# Importing Generated Routes (Only to test the same routes)
list_of_networks_path = "Saved Networks/"

list_of_networks = import_networks(f"{list_of_networks_path}Makati_Route_networks_Mixed.pkl")
# list_of_networks = import_networks(f"{list_of_networks_path}Makati_Route_networks_Mixed_W{walk_distance}.pkl")

In [ ]:
merged_amenities_points_gdf = gpd.read_file('./City Data/Makati City/Makati_point.geojson')
merged_amenities_polygons_gdf= gpd.read_file('./City Data/Makati City/Makati_polygon.geojson')

merged_amenities_polygons_gdf['amenity_points'] = None
merged_amenities_polygons_gdf['area'] = degrees_to_meters(merged_amenities_polygons_gdf['geometry'].area)
makati_area_sum = merged_amenities_polygons_gdf['area'].sum()

In [ ]:
# IMPORT INITIAL NETWORK
initial_network = import_networks(f"{list_of_networks_path}Makati_Initial_Network.pkl")

# IMPORT ZONE NETWORK
graph_networks_of_polygons = import_networks(f"{list_of_networks_path}Makati_Zone_Network.pkl")

add_points_to_graph(initial_network, graph_networks_of_polygons) # Add all points

In [ ]:
# LEVEL 3 - CREATING STOPS TO BE PLACED ON ZONES
graph_of_stops = copy.deepcopy(CITY_GRAPH)
list_of_stops = []
place_stops_on_roads(graph_networks_of_polygons, graph_of_stops, list_of_stops) # Adds stops graph_of_stops, and gets the relevant edges
nid = len(list_of_stops)
if nid in graph_of_stops and graph_of_stops.nodes[nid] == {}:
    graph_of_stops.remove_node(nid)

# Visualize the stops
# stops_map = plot_stops_on_map(list_of_stops_Makati)
# stops_map.save(f"{Makati_map_filepath}stops_map.html")

In [ ]:
# Configuration
max_stops = 20
max_routes = 25
walk_distance = WALKING_DISTANCES[0] # WALKING_DISTANCES -> 300,550,800

route_network, route_graph, reversed_route_network = generate_route_network(list_of_stops, walk_distance, max_stops, max_routes, graph_of_stops, makati_area_sum)